# 11.6 — The Iceberg Metadata Tree

**Chapter 11, §6.1 (lines 478–517 and Figure 11.6), §6.2 (lines 586–615), the maintenance note
at lines 560–570, and Exercise 6.**

Figure 11.6 draws a table as four layers with a pointer on top: a catalog reference to a table
metadata file, which names a snapshot, which names a manifest list, which names manifests, which
name Parquet data files. This notebook builds that tree on disk and walks it, then uses it for
the three things the chapter says it buys — hidden partitioning, time travel, and a maintenance
bill.

> **On the engine.** No Apache Iceberg Spark runtime loads on Spark 4.2: both
> `iceberg-spark-runtime-4.0_2.13:1.11.0` and `-4.1_2.13:1.11.0` fail with
> `IncompatibleClassChangeError` on `connector.catalog.View`, which Spark 4.2 changed from an
> interface into a class. The notebook therefore uses **PyIceberg**, the Python implementation of
> the same specification. The files it writes are ordinary Iceberg; §6.3's whole argument is that
> this should not matter.

In [1]:
import datetime as dt
import json, os, shutil, sqlite3, tempfile
import pyarrow as pa
from pyiceberg.catalog.sql import SqlCatalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, LongType, StringType, TimestampType
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import DayTransform

SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch11-iceberg")
shutil.rmtree(SCRATCH, ignore_errors=True)
WAREHOUSE = os.path.join(SCRATCH, "warehouse")
os.makedirs(WAREHOUSE)

# The catalog. Everything in Figure 11.6 above the metadata file lives here: this SQLite
# database holds one row per table, and that row holds the pointer.
catalog = SqlCatalog("local",
                     uri=f"sqlite:///{os.path.join(WAREHOUSE, 'catalog.db')}",
                     warehouse=f"file://{WAREHOUSE}")
catalog.create_namespace("db")

In [2]:
# Hidden partitioning: the table is partitioned by the day of event_time, and there is no
# event_date column for anyone to maintain or to remember to filter on.
schema = Schema(
    NestedField(1, "event_id", LongType(), required=False),
    NestedField(2, "event_time", TimestampType(), required=False),
    NestedField(3, "payload", StringType(), required=False),
)
spec = PartitionSpec(
    PartitionField(source_id=2, field_id=1000, transform=DayTransform(), name="event_time_day")
)
events = catalog.create_table("db.events", schema=schema, partition_spec=spec)

BASE = dt.datetime(2026, 3, 1)
batch = pa.table(
    {
        "event_id": pa.array(list(range(12)), pa.int64()),
        "event_time": pa.array([BASE + dt.timedelta(days=i % 3, hours=i) for i in range(12)],
                               pa.timestamp("us")),
        "payload": pa.array([f"payload-{i}" for i in range(12)]),
    },
    schema=events.schema().as_arrow(),
)
events.append(batch)
print(events.scan().to_arrow().num_rows, "rows in", events.inspect.files().num_rows, "data files")

12 rows in 3 data files


## 1. The tree on disk

Everything the table is, apart from the catalog row.

In [3]:
TABLE_DIR = os.path.join(WAREHOUSE, "db", "events")


def tree():
    for root, _, names in sorted(os.walk(TABLE_DIR)):
        for name in sorted(names):
            rel = os.path.relpath(os.path.join(root, name), TABLE_DIR)
            print("  ", rel)


tree()

   data/event_time_day=2026-03-01/00000-0-d9836fad-da2b-474a-b6f5-1ad85612ca36.parquet
   data/event_time_day=2026-03-02/00000-1-d9836fad-da2b-474a-b6f5-1ad85612ca36.parquet
   data/event_time_day=2026-03-03/00000-2-d9836fad-da2b-474a-b6f5-1ad85612ca36.parquet
   metadata/00000-0867e3b2-489f-4cde-834b-90835e520e53.metadata.json
   metadata/00001-642caff3-f1eb-4071-9d2f-d832ff5c617a.metadata.json
   metadata/d9836fad-da2b-474a-b6f5-1ad85612ca36-m0.avro
   metadata/snap-2086170185623062070-0-d9836fad-da2b-474a-b6f5-1ad85612ca36.avro


Four kinds of file, and they are the four levels of Figure 11.6 read bottom-up:

* `data/event_time_day=.../*.parquet` — the data, in directories named by the partition
  *transform* rather than by a stored column;
* `metadata/*-m0.avro` — a **manifest**, listing data files with their statistics;
* `metadata/snap-*.avro` — a **manifest list**, one per snapshot, listing the manifests;
* `metadata/*.metadata.json` — the **table metadata file**: schema, partition spec, and the log
  of snapshots.

## 2. What the metadata file says

In [4]:
meta = json.load(open(events.metadata_location.replace("file://", "")))
print("current-snapshot-id:", meta["current-snapshot-id"])
print("schema field ids   :", [(f["id"], f["name"]) for f in meta["schemas"][0]["fields"]])
print("partition spec     :", meta["partition-specs"][0]["fields"])
print("snapshots          :", [(s["snapshot-id"], s["summary"]["operation"]) for s in meta["snapshots"]])

current-snapshot-id: 2086170185623062070
schema field ids   : [(1, 'event_id'), (2, 'event_time'), (3, 'payload')]
partition spec     : [{'source-id': 2, 'field-id': 1000, 'transform': 'day', 'name': 'event_time_day'}]
snapshots          : [(2086170185623062070, 'append')]


The **field ids** are the mechanism behind §6.1's third bullet: a column is identified by `2`, not
by the name `event_time`, so renaming it is a metadata edit that leaves every data file valid.

The partition spec records `source-id: 2` with transform `day` — the partitioning is a property
of the *table*, not a column in the rows.

## 3. Following the pointer down

In [5]:
snap = events.current_snapshot()
print("snapshot       :", snap.snapshot_id)
print("manifest list  :", os.path.basename(snap.manifest_list))
for m in events.inspect.manifests().to_pylist():
    print("  manifest     :", os.path.basename(m["path"]),
          "->", m["added_data_files_count"], "data files")
for f in events.inspect.files().to_pylist():
    print("    data file  :", os.path.relpath(f["file_path"].replace("file://", ""), TABLE_DIR),
          f"{f['record_count']} rows, {f['file_size_in_bytes']} bytes")

snapshot       : 2086170185623062070
manifest list  : snap-2086170185623062070-0-d9836fad-da2b-474a-b6f5-1ad85612ca36.avro
  manifest     : d9836fad-da2b-474a-b6f5-1ad85612ca36-m0.avro -> 3 data files
    data file  : data/event_time_day=2026-03-01/00000-0-d9836fad-da2b-474a-b6f5-1ad85612ca36.parquet 4 rows, 1422 bytes
    data file  : data/event_time_day=2026-03-02/00000-1-d9836fad-da2b-474a-b6f5-1ad85612ca36.parquet 4 rows, 1425 bytes
    data file  : data/event_time_day=2026-03-03/00000-2-d9836fad-da2b-474a-b6f5-1ad85612ca36.parquet 4 rows, 1427 bytes


## 4. The statistics that let files be skipped

§6.1 says manifests carry "partition values, file-level statistics, column minimum and maximum
values, and null counts". They are readable directly.

In [6]:
for f in events.inspect.files().to_pylist():
    m = f["readable_metrics"]
    print(os.path.basename(f["file_path"]),
          "| partition", f["partition"],
          "| event_time in", m["event_time"]["lower_bound"], "..", m["event_time"]["upper_bound"])

00000-0-d9836fad-da2b-474a-b6f5-1ad85612ca36.parquet | partition {'event_time_day': datetime.date(2026, 3, 1)} | event_time in 2026-03-01 00:00:00 .. 2026-03-01 09:00:00
00000-1-d9836fad-da2b-474a-b6f5-1ad85612ca36.parquet | partition {'event_time_day': datetime.date(2026, 3, 2)} | event_time in 2026-03-02 01:00:00 .. 2026-03-02 10:00:00
00000-2-d9836fad-da2b-474a-b6f5-1ad85612ca36.parquet | partition {'event_time_day': datetime.date(2026, 3, 3)} | event_time in 2026-03-03 02:00:00 .. 2026-03-03 11:00:00


A planner holding this does not have to open a file — or list a directory — to know it cannot
contain a row for 3 March.

## 5. Hidden partitioning

The query filters on `event_time`. It says nothing about days, nothing about the partition
column, and it does not need to know the table is partitioned at all.

In [7]:
scan = events.scan(row_filter="event_time >= '2026-03-02T00:00:00'")
print("data files in the table :", events.inspect.files().num_rows)
print("files this scan will open:", len(list(scan.plan_files())))
print("rows returned            :", scan.to_arrow().num_rows)

data files in the table : 3
files this scan will open: 2
rows returned            : 8


Two files out of three. The planner applied the `day` transform to the predicate itself and
discarded the partition that cannot match — which is what §6.2 means by the query being separated
from the physical organization. In 11.3 the same saving required the writer to create an explicit
partition column and every query to remember to filter on it.

## 6. Commits are a pointer swap

Three more appends, and the thing that changes with each one.

In [8]:
def catalog_pointer():
    con = sqlite3.connect(os.path.join(WAREHOUSE, "catalog.db"))
    row = con.execute("SELECT metadata_location FROM iceberg_tables").fetchone()
    con.close()
    return os.path.basename(row[0])


print("catalog now points at:", catalog_pointer())
for day in (0, 1, 2):
    events.append(pa.table({
        "event_id": pa.array([100 + day], pa.int64()),
        "event_time": pa.array([BASE + dt.timedelta(days=day, minutes=30)], pa.timestamp("us")),
        "payload": pa.array([f"late-{day}"]),
    }, schema=events.schema().as_arrow()))
    print("after append      :", catalog_pointer())

catalog now points at: 00001-642caff3-f1eb-4071-9d2f-d832ff5c617a.metadata.json
after append      : 00002-5b806480-b076-4721-9548-2810a2a908a7.metadata.json
after append      : 00003-c9533742-0405-4c0c-81e2-71c5d3a1ee89.metadata.json
after append      : 00004-5282a62a-c5a7-4fa7-9489-54de75b1fb5f.metadata.json


Each commit writes a **new** metadata file and then changes one row of the catalog to point at
it. Nothing already written is modified, and the transition is a single compare-and-swap on that
row — the atomic operation Exercise 2(b) asks for, and the reason Iceberg needs no directory
rename in a flat key space.

## 7. Time travel

Every metadata file that was ever current is still on disk, and every snapshot it named is still
in the snapshot log.

In [9]:
for s in events.inspect.snapshots().to_pylist():
    rows = events.scan(snapshot_id=s["snapshot_id"]).to_arrow().num_rows
    print(f"snapshot {s['snapshot_id']:>21}  {s['operation']:<8} {rows:>3} rows")

snapshot   2086170185623062070  append    12 rows
snapshot   8350903752552064215  append    13 rows
snapshot   3299577799742167110  append    14 rows
snapshot   4763506187230606830  append    15 rows


This is the table of §6.2 — `VERSION AS OF` in SQL, a snapshot id here, and the same mechanism
underneath: an older snapshot names an older manifest list, which names files that were never
deleted.

## 8. The maintenance bill

Three late appends landed one row each in days that already had a file, so the three partitions
now hold two files apiece. This is the small-file problem of 11.3, arriving through a different
door: frequent commits rather than a high-cardinality partition key.

In [10]:
def file_stats():
    files = events.inspect.files().to_pylist()
    total = sum(f["file_size_in_bytes"] for f in files)
    return len(files), total // max(len(files), 1)


n, mean = file_stats()
print(f"{n} data files, mean {mean} bytes")

6 data files, mean 1389 bytes


In [11]:
# Compaction: rewrite the current contents as few large files. PyIceberg has no
# rewrite_data_files action yet, so the same effect is obtained by overwriting the table
# with its own scan -- which is what the action does underneath.
events.overwrite(events.scan().to_arrow())
n, mean = file_stats()
print(f"after compaction: {n} data files, mean {mean} bytes")
print("snapshots now:", len(events.metadata.snapshots))
print("parquet files still on disk:",
      sum(1 for r, _, fs in os.walk(TABLE_DIR) for f in fs if f.endswith(".parquet")))

after compaction: 3 data files, mean 1438 bytes
snapshots now: 6
parquet files still on disk: 9


Compaction halved the number of files the table uses and *increased* the number on disk, because
the originals are still referenced by earlier snapshots. Storage goes up before it goes down,
which is the arithmetic Exercise 6(b) asks for.

## 9. Snapshot expiration, and the end of time travel

In [12]:
old = events.metadata.snapshots[0].snapshot_id
keep = events.current_snapshot().snapshot_id

events.maintenance.expire_snapshots().by_ids(
    [s.snapshot_id for s in events.metadata.snapshots if s.snapshot_id != keep]
).commit()

print("snapshots retained:", [s.snapshot_id for s in events.metadata.snapshots])
print("parquet files on disk:",
      sum(1 for r, _, fs in os.walk(TABLE_DIR) for f in fs if f.endswith(".parquet")))

snapshots retained: [157275174012618159]
parquet files on disk: 9


In [13]:
try:
    events.scan(snapshot_id=old).to_arrow()
except ValueError as exc:
    print("time travel to the expired snapshot:", exc)
print("the current table is unaffected:", events.scan().to_arrow().num_rows, "rows")

time travel to the expired snapshot: Snapshot not found: 2086170185623062070
the current table is unaffected: 15 rows


Exercise 6(a) asks for exactly this distinction, and the two cells above separate the halves.
Expiring the snapshots removed the *metadata* that identified those versions — the query now
fails on "Snapshot not found" rather than on a missing file. The **data files are all still
there**: nine Parquet files on disk for a table that uses three.

That gap is the third maintenance job of §6.1's note box.

In [14]:
on_disk = {os.path.join(r, f) for r, _, fs in os.walk(TABLE_DIR) for f in fs
           if f.endswith(".parquet")}
referenced = set()
for s in events.metadata.snapshots:
    for f in events.scan(snapshot_id=s.snapshot_id).plan_files():
        referenced.add(f.file.file_path.replace("file://", ""))

print(f"{len(on_disk)} parquet files on disk, {len(referenced)} reachable from a retained snapshot")
print(f"{len(on_disk - referenced)} orphans, holding "
      f"{sum(os.path.getsize(f) for f in on_disk - referenced):,} bytes")

9 parquet files on disk, 3 reachable from a retained snapshot
6 orphans, holding 8,339 bytes


Nothing in the table points at those files any more, and nothing will delete them either:
snapshot expiration, file compaction and orphan-file cleanup are three separate jobs, and only
the third one is looking for these. A time-travel query can therefore fail for two different
reasons — the snapshot is no longer named, or its files have been swept — and the storage bill
keeps running until the sweep happens.

## Conclusion

* An Iceberg table is a tree: catalog row → metadata file → manifest list → manifests → data
  files. Every level above the data is metadata that can be rewritten without touching a byte of
  Parquet.
* A commit writes new metadata and swaps one pointer. No rename, no directory listing, and no
  window in which half the change is visible.
* Because the partition transform is stored in the table metadata, a query filters on the raw
  column and the planner derives the partition predicate — two files opened out of three, with
  nothing in the query naming a partition.
* Field ids, not names, identify columns, which is what makes a rename a metadata-only change.
* The history that makes time travel possible is stored, and it is not free. Compaction adds
  files before anything removes them; expiring a snapshot ends a version's life but deletes no
  bytes; and the bytes are only reclaimed by a third job that looks for files no retained
  snapshot can reach. The chapter's note box lists all three for that reason.